In [1]:
# Install required libraries (run once)
!pip install transformers torch


In [5]:
# Import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


C:\Users\Namrata\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Load pre-trained model and tokenizer from Hugging Face
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
print("✅ Model loaded successfully!")


Loading weights: 100%|█████████████████████████████████████████████████████████████| 293/293 [00:00<00:00, 6634.59it/s]


✅ Model loaded successfully!


In [7]:
def chatbot():
    print("🤖 Chatbot: Hello! Type 'exit' to stop.")
    chat_history_ids = None

    while True:
        user_input = input("👤 You: ")

        if user_input.lower() in ["exit", "quit"]:
            print("🤖 Chatbot: Goodbye!")
            break

        new_input_ids = tokenizer.encode(
            user_input + tokenizer.eos_token,
            return_tensors='pt'
        )

        bot_input_ids = new_input_ids if chat_history_ids is None else torch.cat(
            [chat_history_ids, new_input_ids], dim=-1
        )

        # ✅ Now works properly
        attention_mask = bot_input_ids.ne(tokenizer.pad_token_id).long()

        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=200,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.6
        )

        response = tokenizer.decode(
            chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        print(f"🤖 Chatbot: {response}\n")

In [ ]:
chatbot()

🤖 Chatbot: Hello! Type 'exit' to stop.


👤 You:  Hello


🤖 Chatbot: ..



👤 You:  What is Artificial Intelligence?
